# 02 — Prepared dataset audit

This notebook validates the prepared Referee tree: manifest fields, split isolation, label/mask consistency, and 512×512 PNG pixel hashes. It does not load a model or compute recognition metrics.

The audit distinguishes reviewed labels from encoded ATYPICT proxies. A proxy agreement is not reviewed game accuracy.


In [ ]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
import json

import pandas as pd
from IPython.display import display

from referee_evaluation.dataset_audit import audit_prepared

REPOSITORY_ROOT = Path.cwd().resolve()
while not (REPOSITORY_ROOT / 'models' / 'referee' / 'data').is_dir():
    REPOSITORY_ROOT = REPOSITORY_ROOT.parent
MODEL_PROJECT = REPOSITORY_ROOT / 'models' / 'referee'


## Options

Choose the same profile used for preparation. `sample` is a wiring/smoke profile; `train` is the real profile after authorization.


In [ ]:
PROFILE = 'sample'          # 'sample' or 'train'
DATA_ROOT = MODEL_PROJECT / 'data' / '.local' / 'data'
PREPARED_DIR = DATA_ROOT / PROFILE / 'prepared'
VERIFY_IMAGES = True
WRITE_AUDIT = False
AUDIT_PATH = MODEL_PROJECT / 'checkpoints' / 'referee-evaluation' / PROFILE / 'dataset-audit.json'

print('Prepared:', PREPARED_DIR)
print('Audit output:', AUDIT_PATH)


## Readiness

If this reports missing files, run `01_dataset_pipeline.ipynb` for the same profile first. The audit never downloads data.


In [ ]:
summary_path = PREPARED_DIR.parent / 'dataset-summary.json'
labels_path = PREPARED_DIR.parent / 'labels.json'
manifests = sorted((PREPARED_DIR / 'manifests').glob('*.jsonl'))
print('Summary present:', summary_path.is_file())
print('Labels present:', labels_path.is_file())
print('Manifests:', [path.name for path in manifests])
if not summary_path.is_file() or not manifests:
    print('Prepared data is incomplete. Run notebook 01 for', PROFILE, 'first.')


## Audit

The function raises on a malformed manifest, a split leak, a mismatched hash, or a missing image. Warnings and limits are returned explicitly.


In [ ]:
if not summary_path.is_file() or not manifests:
    report = None
    print('Audit skipped: prepared data is absent.')
else:
    report = audit_prepared(PREPARED_DIR, verify_images=VERIFY_IMAGES)
    if WRITE_AUDIT:
        AUDIT_PATH.parent.mkdir(parents=True, exist_ok=True)
        AUDIT_PATH.write_text(json.dumps(report, indent=2) + '\n', encoding='utf-8')
        print('Wrote', AUDIT_PATH)


## Counts

Read the known/positive/negative/unknown counts per channel, not only the total row count.


In [ ]:
if report is not None:
    rows = []
    for split, entry in report['splits'].items():
        rows.append({
            'split': split,
            'rows': entry['rows'],
            'images_ok': entry['images_ok'],
            'text_known': entry['labels']['text']['known'],
            'text_positive': entry['labels']['text']['positive'],
            'number_known': entry['labels']['number']['known'],
            'number_positive': entry['labels']['number']['positive'],
            'number_unknown': entry['labels']['number']['unknown'],
        })
    display(pd.DataFrame(rows).set_index('split'))
    display(pd.DataFrame.from_dict(report['totals'], orient='index', columns=['value']))
    for warning in report['warnings']:
        print('-', warning)


## Next

Open `03_training_and_checkpoint.ipynb` to inspect the supplied checkpoint before running validation evaluation.
